# 12 · Real crack: VisionCrack vs. ruler (GT)

Is VisionCrack's reading of a real crack correct, and how large is the error? The 10 photos from notebook 11 are compared with the
crack-width ruler reading (GT = 0.45 mm) at the same measuring point, for the five options of notebook 11, in the layout of the paper's
Table 1. The errors are then set against VisionCrack's error on the printed targets of Table 1.

| | |
|---|---|
| **Inputs** | `data/raw/images/VisionCrack_*.JPG` (10 photos); `data/raw/table1_et/` (paper Table 1, for the comparison) |
| **Outputs** | `results/tables/field_table.csv/.tex`, `field_vs_table1.csv`; `results/figures/field_result.png` |
| **Runtime** | ~30 s |

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import visioncrack as vc
from visioncrack import io, field
from visioncrack.constants import CLIP_Z, LABEL_WIDTH_MM, ROTATION_MAX_DEG
from visioncrack.metrics import half_up
from visioncrack.width import binarize, blur_gray
pd.set_option('display.width', 200)
print('visioncrack', vc.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
FIG = io.results_dir('figures'); TAB = io.results_dir('tables')

files = sorted((io.data_dir() / 'images').glob('VisionCrack_*.JPG'))
assert len(files) == 10, f'expected 10 photos, found {len(files)}'
NAMES = [f'VisionCrack {i + 1}' for i in range(len(files))]

POINT_MM = (7.0, 9.0)      # measuring point: the 2-mm part of the crack 7–9 mm from the label frame, where the ruler was read
GT_MM = 0.45               # crack-width ruler reading by the author at the measuring point

# five ways to get one width for the measuring point (name: arguments of field.point_values)
OPTIONS = {
    'middle row':           dict(rule='subpixel', scale='barcode', stat='middle'),
    'median':               dict(rule='subpixel', scale='barcode', stat='median'),
    'mean':                 dict(rule='subpixel', scale='barcode', stat='mean'),
    'Table 1 rule, > 2 px': dict(rule='paper', scale='label', stat='clipped', min_run=2),
    'Table 1 rule, > 6 px': dict(rule='paper', scale='label', stat='clipped', min_run=6),
}
rs = [field.measure_field(f) for f in files]
V = pd.DataFrame({name: [field.point_values(r, points=(POINT_MM,), **kw)[0] for r in rs] for name, kw in OPTIONS.items()},
                 index=NAMES)
print(len(rs), 'photos measured; empty values:', int(V.isna().sum().sum()))

## 1 · Table F: real crack in Table 1 layout, one column per option

**GT** first, one row per photo, then the **mean**, **STD** (population, ddof = 0) and **|mean − GT|**.
Values rounded half-up to 3 decimals, as in the paper.

In [ ]:
pr = V.round(3)
mean_, std_ = half_up(pr.mean(), 3), half_up(pr.std(ddof=0), 3)
stats = pd.DataFrame({'VisionCrack mean': mean_, 'VisionCrack STD': std_, 'VisionCrack |mean − GT|': np.abs(mean_ - GT_MM)},
                     index=V.columns).T
tab = pd.concat([pd.DataFrame([[GT_MM] * V.shape[1]], columns=V.columns, index=['GT']), pr, stats])
tab.to_csv(TAB / 'field_table.csv', float_format='%.4f', na_rep='')
tab.round(4).to_latex(TAB / 'field_table.tex', float_format='%.3f', na_rep='')
tab.round(4).astype(object).where(tab.notna(), '')

## 2 · How large is the error?

In [ ]:
px_mm = np.mean([r.scale for r in rs])
err = V - GT_MM
E = pd.DataFrame({
    'mean (mm)': V.mean(),
    'mean − GT (mm)': err.mean(),
    '|mean − GT| % of GT': 100 * err.mean().abs() / GT_MM,
    '|mean − GT| (px)': err.mean().abs() / px_mm,
    'STD between photos (mm)': V.std(ddof=0),
    'largest |photo − GT| (mm)': err.abs().max(),
})
print(f'GT = {GT_MM} mm, pixel ≈ {px_mm:.4f} mm')
E.round(4)

## 3 · Compared with VisionCrack on the printed targets (paper Table 1)

Table 1 is recomputed from `data/raw/table1_et/` as in notebook 05 (checked against the paper), with the paper's 6-px filter and with
2 px, to show that the 2-px filter does not change Table 1. *less / close / larger* = below 90 %, within ±10 %, above 110 % of the paper's value.

In [ ]:
from visioncrack.metrics import table1_summary
caps, _ = vc.load_table1()
roi = vc.preview_roi((2160, 3840), (430, 932))
def table1(min_run):
    t1 = pd.DataFrame([vc.measure_capture(vc.read_rgb(p), b, roi, min_run=min_run).widths_mm for p, b in zip(caps.path, caps.bbox)])
    s = table1_summary(t1.round(3).to_numpy(), np.array(vc.ET_GT_MM))
    return half_up(s.loc['std'].iloc[:5], 3), np.abs(half_up(s.loc['mean'].iloc[:5], 3) - np.array(vc.ET_GT_MM))
std6, bias6 = table1(6)
std2, bias2 = table1(2)
assert np.allclose(std6, [0.005, 0.000, 0.013, 0.013, 0.012]), 'Table 1 STD differs from the paper'
assert np.allclose(std2, std6) and np.allclose(bias2, bias6, atol=0.002), 'the 2-px filter changed Table 1'
print(f'✓ Table 1 is the same with the 6-px and the 2-px filter: STD {std6.mean():.4f} mm, |mean − GT| {bias6.mean():.4f} mm (average of ET-1…ET-5)')

def verdict(x, ref):
    return 'less' if x < 0.9 * ref else ('close' if x <= 1.1 * ref else 'larger')
cmp = pd.DataFrame({'STD (mm)': V.std(ddof=0), '|mean − GT| (mm)': (V.mean() - GT_MM).abs()})
cmp.loc['Paper Table 1 · VisionCrack, printed targets'] = [std6.mean(), bias6.mean()]
cmp['STD vs paper'] = [verdict(x, std6.mean()) for x in cmp['STD (mm)']]
cmp['|mean − GT| vs paper'] = [verdict(x, bias6.mean()) for x in cmp['|mean − GT| (mm)']]
cmp.iloc[-1, 2:] = '—'
cmp = cmp.iloc[[-1] + list(range(len(cmp) - 1))]
cmp.to_csv(TAB / 'field_vs_table1.csv', float_format='%.4f')
cmp.round(4)

## 4 · Figure

In [ ]:
COL = dict(zip(V.columns, ['tab:gray', 'red', 'blue', 'tab:purple', 'tab:orange']))
MK = dict(zip(V.columns, ['^', 'o', 's', 'D', 'v']))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.2, 1]})
x = np.arange(1, len(V) + 1)
a1.axhline(GT_MM, color='green', ls='--', lw=2.5, label=f'Ground Truth (ruler) = {GT_MM} mm')
for c in V.columns:
    a1.plot(x, V[c].values, marker=MK[c], color=COL[c], lw=1.6, ms=7, label=c)
a1.set_xticks(x); a1.set_xlabel('Photo', fontsize=13, fontweight='bold')
a1.set_ylabel('Estimated Crack Width (mm)', fontsize=13, fontweight='bold')
a1.grid(alpha=.4); a1.legend(fontsize=9, ncol=2); a1.set_title('Each photo', fontsize=13, fontweight='bold')
xo = np.arange(V.shape[1])
a2.axhline(GT_MM, color='green', ls='--', lw=2.5, label='Ground Truth')
for i, c in enumerate(V.columns):
    a2.errorbar([i], [V[c].mean()], yerr=[V[c].std(ddof=0)], fmt=MK[c], color=COL[c], capsize=6, ms=9, lw=2)
a2.set_xticks(xo, [c.replace('Table 1 rule, ', 'Table 1\n') for c in V.columns], fontsize=10)
a2.set_ylabel('Averaged Crack Width (mm)', fontsize=13, fontweight='bold'); a2.set_ylim(a1.get_ylim())
a2.grid(alpha=.4); a2.legend(fontsize=9); a2.set_title('Mean ± STD of 10 photos', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(FIG / 'field_result.png', dpi=300); plt.show()

## 5 · Summary

In [ ]:
print(f'Real crack, one measuring point (7–9 mm from the label frame), 10 auto-captured photos, GT = {GT_MM} mm (crack-width ruler).')
print(f'Paper Table 1, VisionCrack on printed targets: STD {std6.mean():.4f} mm, |mean − GT| {bias6.mean():.4f} mm.\n')
for c in V.columns:
    print(f'{c:22s}: {V[c].mean():.3f} mm, error {err[c].mean():+.3f} mm ({100 * err[c].mean() / GT_MM:+.1f} %), '
          f'STD {V[c].std(ddof=0):.4f} mm  → STD {cmp.loc[c, "STD vs paper"]}, error {cmp.loc[c, "|mean − GT| vs paper"]} than the paper')

### Notes
* GT is one reading with a crack-width ruler (steps of about 0.05 mm), so it is less exact than the printed widths of Table 1.
  An error below one ruler step cannot be told apart from the ruler's own reading error.
* One crack, one phone (3024 × 4032 photos), one session; the label was not moved between photos. This shows that the code works on a
  real crack and repeats; it is not a full field study.